<div dir="rtl">

# الدرس 20: أشجار القرار

دفتر الكود لهذا الدرس. شغّل الخلايا بالترتيب من الأعلى.

</div>

<div dir="rtl">

### جيني بيدنا

</div>

In [ ]:
import numpy as np
import pandas as pd

df = pd.DataFrame({
    "young":  [1, 1, 1, 0, 0, 1, 0, 0, 0, 0],
    "card":   [1, 1, 0, 1, 1, 0, 1, 0, 0, 0],
    "bought": [1, 1, 1, 1, 1, 0, 0, 0, 0, 0],
})

def gini(labels):
    p = np.bincount(labels, minlength=2) / len(labels)
    return 1 - np.sum(p ** 2)

def split_gain(df, feature, target="bought"):
    parent = gini(df[target].values)
    weighted = 0.0
    for value, part in df.groupby(feature):
        weighted += len(part) / len(df) * gini(part[target].values)
    return parent - weighted

for f in ["young", "card"]:
    print(f"gain of '{f}': {split_gain(df, f):.3f}")

<div dir="rtl">

### نفس الشجرة بمكتبة سايكت ليرن

</div>

In [ ]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(df[["young", "card"]], df["bought"])
print(export_text(tree, feature_names=["young", "card"]))

<div dir="rtl">

### عمق الشجرة والإفراط في التعلّم

</div>

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

depths = range(1, 16)
tr_acc, te_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    tr_acc.append(t.score(X_tr, y_tr))
    te_acc.append(t.score(X_te, y_te))

plt.figure(figsize=(6, 3.4))
plt.plot(depths, tr_acc, "o-", lw=2, label="train")
plt.plot(depths, te_acc, "o-", lw=2, label="test")
plt.xlabel("max_depth"); plt.ylabel("accuracy")
plt.legend(); plt.grid(True)
plt.show()

full = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
print("unlimited tree: depth", full.get_depth(), "| leaves", full.get_n_leaves(),
      "| train", full.score(X_tr, y_tr), "| test", round(full.score(X_te, y_te), 3))

<div dir="rtl">

### أهمية الميزات

</div>

In [ ]:
from sklearn.datasets import load_breast_cancer

names = load_breast_cancer().feature_names
t3 = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
top = np.argsort(t3.feature_importances_)[::-1][:5]
for i in top:
    print(f"{names[i]:25} {t3.feature_importances_[i]:.3f}")

<div dir="rtl">

## تمرين: السؤال الثاني في كل فرع

في الفرع الذي فيه أصحاب البطاقات: خمسة زبائن، أربعة اشتروا.

1. احسب مكسب سؤال العمر داخل هذا الفرع.
2. وفي الفرع الآخر، بدون بطاقات: احسب مكسب سؤال العمر.
3. في أي فرع كان سؤال العمر أكثر فائدة؟
4. تحقق بالكود بدالة المكسب، على كل فرع وحده.

</div>

In [ ]:
card_yes = df[df["card"] == 1]
card_no = df[df["card"] == 0]
# احسب مكسب سؤال العمر في كل فرع